# Notebook 8 / R8-K5 - WasteNet-256K Teacher-Assistant KD

Port K6 R8 ke protokol 5 kelas (TrashNet-K5, kelas `trash` dihapus). Melatih WasteNet-256K (student, 160px) dengan teacher-assistant knowledge distillation dari Focus-RCNet R2-K5 (assistant, 380px) pada seed yang sama, di-init dari checkpoint R6-K5 WasteNet-256K CE (stage-2 fine-tune).
R8-K5 menjawab apakah assistant berkapasitas-menengah (Focus-RCNet ~520K) bisa menjembatani capacity gap yang membuat direct big-teacher KD (R7-K5) gagal -- yaitu mengangkat student di atas R6-K5 no-KD floor.

> Catatan K5 vs K6: di K6, R2 = Focus-RCNet hasil *direct KD* (teacher-assistant selection). Di K5, R2 = Focus-RCNet **CE baseline** (`focus_rcnet_ce_r2_final_seed_*_best.pth`). Jadi assistant R8-K5 adalah Focus-RCNet CE, bukan KD-trained. Arsitektur Focus-RCNet identik dengan K6 sehingga checkpoint memuat bersih.


## Rules

- Gunakan `split_manifest_seed_*.csv` dari R0-K5 dan wajib verifikasi `dataset_name=TrashNet-K5`, `dataset_version=2`, `num_classes=5`, `trash` absen.
- Student init **wajib** dari checkpoint R6-K5 WasteNet-256K CE final pada seed yang sama (stage-2 fine-tune, bukan random init).
- Assistant untuk preset KD **wajib** dari R2-K5 Focus-RCNet CE final pada seed yang sama.
- Pilot R8-K5 hanya memakai validation set (seed 42) untuk memilih `temperature`, `alpha`, dan `stage2_lr`.
- Independent test set hanya dievaluasi ketika `RUN_PHASE = "final"` setelah winner dibekukan.
- Student & assistant menerima augmentasi acak yang **sama** sebelum dibuat view 160px (student) dan 380px (assistant).
- Hyperparameter non-KD (student IMG_SIZE=160, assistant IMG_SIZE=380, 100 epoch final) dibekukan identik dengan K6 R8 agar 5-vs-6 tetap comparable. `stage2_lr` di-pilih lewat pilot (bagian dari grid R5).
- Final R8-K5 dijalankan 5 seed dengan konfigurasi yang sudah dibekukan.


## R8-K5 Pilot - Diagnostic Round (carry R5-K5 lesson, 2026-06-22)

R8-K5 = 256K versi dari R5-K5 (yang gagal mengangkat student 128K). Dua pelajaran mahal dari R5-K5 langsung dipakai di sini, jadi pilot R8 **mulai dari ronde diagnostik** (bukan grid K6 mentah):
- **(a) LR-shock:** `stage2_lr` 1e-3/5e-4 (warisan K6) menendang checkpoint yang sudah konvergen keluar dari optimum-nya di epoch-1. Di R5-K5 hanya CE fine-tune `lr=1e-4` yang akhirnya melewati base. Maka R8 menyapu **LR sangat rendah {1e-5, 1e-4}**, BUKAN 5e-4 K6.
- **(b) Mismatch resolusi:** assistant melihat 380px sementara student cuma 160px -> soft-target meng-encode detail yang student tak bisa jawab. Maka R8 menguji **resolusi assistant disamakan (av160)** vs mismatch asli (av380).

Faktorial pada bobot KD nyata (`alpha=0.5`, `T=4`): `stage2_lr {1e-5, 1e-4}` x `{CE control, TA-KD av160 (matched), TA-KD av380 (mismatch)}` = 6 config, seed 42, 60 ep tanpa ES.

- `assistant_img_size=160` -> assistant diberi input 160px (sama dgn student) = resolusi match.
- `assistant_img_size=380` -> setup asli (mismatch).
- **Kriteria sukses cuma satu: ada config yang tembus > 0.7933 (base R6-K5 256K no-KD FINAL seed-42 ckpt yg di-load R8, val @ep69; BUKAN 0.8184 yg itu run pilot-convergence R6 yg beda).** Kalau di LR rendah + resolusi-match + alpha tinggi pun tak tembus -> hasil negatif solid, lapor apa adanya (jangan p-hack di seed-42 val n=358).

> Ekspektasi (dari R4/R5/R7-K5): kemungkinan besar wash lagi -- seluruh student tier nyangkut ~0.80-0.81 lepas dari ukuran (128K/256K) atau jenis KD. Tapi R8 = satu-satunya config K6 yang mendekati signifikan (+1.0pp p=0.099), jadi ini tes penentu untuk 256K.

Edit `PILOT_SWEEP` di Config buat ganti config (pilihan di `PILOT_CONFIGS`). Untuk final (setelah pilot): isi `FINAL_SWEEP` (TA-KD winner + CE control), `RUN_PHASE='final'`, jalankan 5 seed (tiap Run All = kedua config sekaligus, test-set ON).


In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import copy
import hashlib
import json
import os
import random
import time
import warnings

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports ready")

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

class Config:
    EXPERIMENT_ID = "R8"
    EXPERIMENT_NAME = "R8-K5 WasteNet-256K Teacher-Assistant KD"
    PLATFORM = "Kaggle Notebooks"
    FRAMEWORK = "PyTorch"
    USE_AMP = True
    MULTI_GPU = False

    # Non-KD hyperparameters frozen identical to K6 R8 so 5-vs-6 stays comparable.
    SEED = 3407  # Pilot: seed 42 only. Final seeds: 42, 123, 777, 2026, 3407
    RUN_PHASE = "final"  # "pilot" = PILOT_SWEEP on seed 42; "final" = FINAL_SWEEP (TA-KD + CE control) per seed

    DATASET_NAME = "TrashNet-K5"
    DATASET_VERSION = 2
    DATASET_DIR = Path(os.environ.get(
        "TRASHNET_K5_DATASET_DIR",
        "/kaggle/input/datasets/kholiqbudiman/trashnet-k5-waste-classification",
    ))

    R0_DIR_CANDIDATES = [
        Path(os.environ.get("R0_K5_DATA_PROTOCOL_DIR", "")),
        Path("/kaggle/working/final_research_kd_k5/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-k5-data-protocol-setup/final_research_kd_k5/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-k5-data-protocol-setup/r0_data_protocol"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-k5-data-protocol-setup/final_research_kd_k5/r0_data_protocol"),
        Path.cwd() / "final_research_kd_k5" / "r0_data_protocol",
    ]
    R2_ASSISTANT_ROOT_CANDIDATES = [
        Path(os.environ.get("R2_K5_ASSISTANT_DIR", "")),
        Path("/kaggle/working/final_research_kd_k5/runs/final/R2"),
        Path("/kaggle/input/output-notebook2-r2-k5-focus-rcnet-ce-final/final_research_kd_k5/runs/final/R2"),
        Path("/kaggle/input/output-notebook2-r2-k5-focus-rcnet-ce-final/runs/final/R2"),
        Path("/kaggle/input/notebook2-r2-k5-focus-rcnet-ce-final/final_research_kd_k5/runs/final/R2"),
        Path("/kaggle/input/notebook2-r2-k5-focus-rcnet-ce-final/runs/final/R2"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook2-r2-k5-focus-rcnet-ce-final/final_research_kd_k5/runs/final/R2"),
        Path("/kaggle/input/r2-k5-final/final_research_kd_k5/runs/final/R2"),
        Path.cwd() / "final_research_kd_k5" / "runs" / "final" / "R2",
    ]
    R6_STUDENT_ROOT_CANDIDATES = [
        Path(os.environ.get("R6_K5_STUDENT_DIR", "")),
        Path("/kaggle/working/final_research_kd_k5/runs/final/R6"),
        Path("/kaggle/input/output-notebook6-r6-k5-wastenet-256k-ce-baseline/final_research_kd_k5/runs/final/R6"),
        Path("/kaggle/input/output-notebook6-r6-k5-wastenet-256k-ce-baseline/runs/final/R6"),
        Path("/kaggle/input/notebook6-r6-k5-wastenet-256k-ce-baseline/final_research_kd_k5/runs/final/R6"),
        Path("/kaggle/input/notebook6-r6-k5-wastenet-256k-ce-baseline/runs/final/R6"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook6-r6-k5-wastenet-256k-ce-baseline/final_research_kd_k5/runs/final/R6"),
        Path("/kaggle/input/r6-k5-final/final_research_kd_k5/runs/final/R6"),
        Path.cwd() / "final_research_kd_k5" / "runs" / "final" / "R6",
    ]

    DEFAULT_OUTPUT_ROOT = (
        Path("/kaggle/working/final_research_kd_k5/runs")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "final_research_kd_k5" / "runs"
    )
    OUTPUT_ROOT = Path(os.environ.get("R8_K5_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))

    MODEL_NAME = "WasteNet-256K"
    VARIANT_ID = "wastenet_256k"
    ASSISTANT_MODEL_NAME = "Focus-RCNet"
    ASSISTANT_SOURCE_VARIANT = "R2-K5 Focus-RCNet CE baseline"

    STUDENT_IMG_SIZE = 160  # WasteNet-256K design size (matches K6 R8 / R6-K5); do NOT change.
    ASSISTANT_IMG_SIZE = 380  # Focus-RCNet R2-K5 trained at 380.
    FINAL_EPOCHS = 100
    PILOT_EPOCHS = 60
    EPOCHS = PILOT_EPOCHS if RUN_PHASE == "pilot" else FINAL_EPOCHS
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    SCHEDULER = "CosineAnnealingLR"

    # --- KD grid lookup (K6 R8 grid kept for reference / config provenance) ---
    PILOT_GRID_TEMPERATURES = [2.0, 4.0, 6.0]
    PILOT_GRID_ALPHAS = [0.05, 0.1, 0.3]
    PILOT_STAGE2_LRS = [0.001, 0.0005, 0.0001, 1e-05]
    PILOT_ANCHOR = {"temperature": 4.0, "alpha": 0.5}
    PILOT_CONFIGS = {
        # TA-KD (Focus-RCNet assistant -> WasteNet-256K student), stage-2 fine-tune from R6-K5 init.
        "ta_t4p0_a0p05_lr0p0005": {"use_kd": True,  "temperature": 4.0, "alpha": 0.05, "stage2_lr": 0.0005, "priority": 0},
        "ta_t2p0_a0p3_lr0p0005":  {"use_kd": True,  "temperature": 2.0, "alpha": 0.3,  "stage2_lr": 0.0005, "priority": 1},
        "ta_t2p0_a0p05_lr0p0005": {"use_kd": True,  "temperature": 2.0, "alpha": 0.05, "stage2_lr": 0.0005, "priority": 2},
        "ta_t4p0_a0p1_lr0p0005":  {"use_kd": True,  "temperature": 4.0, "alpha": 0.1,  "stage2_lr": 0.0005, "priority": 3},
        "ta_t4p0_a0p05_lr0p001":  {"use_kd": True,  "temperature": 4.0, "alpha": 0.05, "stage2_lr": 0.001,  "priority": 4},
        "anchor_t4_a0p5_lr0p0005":{"use_kd": True,  "temperature": 4.0, "alpha": 0.5,  "stage2_lr": 0.0005, "priority": 5},
        # CE fine-tune control (no KD, no assistant): isolates assistant benefit vs. plain stage-2 fine-tune.
        "ce_finetune_lr0p001":    {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 0.001,  "priority": 6},
        "ce_finetune_lr0p0005":   {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 0.0005, "priority": 7},
        # --- Diagnostic round (carry R5-K5 lesson, 2026-06-22). R5-K5 showed two faults
        # that the K6 grid hides: (a) stage2_lr too high (1e-3/5e-4) warm-restarts a
        # converged ckpt off its optimum at epoch 1; (b) the assistant sees 380px detail the
        # 160px student cannot, so soft targets conflict (higher alpha made it worse). So
        # sweep tiny LR x matched assistant resolution at REAL KD weight. assistant_img_size:
        # 160 == matched to the student, 380 == original mismatch. Success = a config that
        # beats the R6-K5 256K no-KD base (the R6 FINAL seed-42 ckpt R8 loads, val 0.7933 @ep69 -- NOT 0.8184, which was R6's separate pilot-convergence run).
        "ce_finetune_lr1em4":      {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 0.0001, "priority": 8},
        "ce_finetune_lr1em5":      {"use_kd": False, "temperature": None, "alpha": None, "stage2_lr": 1e-05,  "priority": 9},
        "ta_t4_a0p5_lr1em4_av160": {"use_kd": True,  "temperature": 4.0,  "alpha": 0.5,  "stage2_lr": 0.0001, "assistant_img_size": 160, "priority": 10},
        "ta_t4_a0p5_lr1em4_av380": {"use_kd": True,  "temperature": 4.0,  "alpha": 0.5,  "stage2_lr": 0.0001, "assistant_img_size": 380, "priority": 11},
        "ta_t4_a0p5_lr1em5_av160": {"use_kd": True,  "temperature": 4.0,  "alpha": 0.5,  "stage2_lr": 1e-05,  "assistant_img_size": 160, "priority": 12},
        "ta_t4_a0p5_lr1em5_av380": {"use_kd": True,  "temperature": 4.0,  "alpha": 0.5,  "stage2_lr": 1e-05,  "assistant_img_size": 380, "priority": 13},
    }
    # Diagnostic sweep (seed 42 only). All configs run in ONE Run All; see driver cell.
    # Factorial: stage2_lr {1e-5, 1e-4} x {CE control, TA-KD av160 matched, TA-KD av380 mismatch}.
    PILOT_SWEEP = [
        "ce_finetune_lr1em5",
        "ta_t4_a0p5_lr1em5_av160",
        "ta_t4_a0p5_lr1em5_av380",
        "ce_finetune_lr1em4",
        "ta_t4_a0p5_lr1em4_av160",
        "ta_t4_a0p5_lr1em4_av380",
    ]
    PILOT_PRESET = PILOT_SWEEP[0]  # representative; the sweep overrides per config

    # --- Final configs (FROZEN from the diagnostic pilot, 2026-06-22). ---
    # Both run in ONE Run All per seed with the test set evaluated: the R5 method (best
    # TA-KD config) PLUS a matched CE fine-tune control, so the KD-vs-noKD gap is measured
    # on the independent test set across 5 seeds. Diagnostic verdict: KD did not beat the
    # control on val, so the control is reported alongside, not replaced. Both keys must
    # exist in PILOT_CONFIGS.
    # FROZEN from the R8-K5 diagnostic pilot (2026-06-22, seed 42). Winner-among-KD =
    # ta_t4_a0p5_lr1em4_av380 (val 0.7961, peak ep13); av160 matched-res was HARMFUL
    # (peaked ep1 then decayed). Matched CE control = ce_finetune_lr1em4 (same LR, 0.7961).
    FINAL_SWEEP = ["ta_t4_a0p5_lr1em4_av380", "ce_finetune_lr1em4"]

    _active_preset = PILOT_PRESET if RUN_PHASE == "pilot" else FINAL_SWEEP[0]
    USE_KD = PILOT_CONFIGS[_active_preset]["use_kd"]
    KD_TEMPERATURE = PILOT_CONFIGS[_active_preset]["temperature"]
    KD_ALPHA = PILOT_CONFIGS[_active_preset]["alpha"]
    STAGE2_LR = PILOT_CONFIGS[_active_preset]["stage2_lr"]

    CHECKPOINT_METRIC = "best_val_accuracy"
    PILOT_EARLY_STOPPING = False  # K6 R8 confirmation pilots ran full epochs without ES.
    EARLY_STOPPING = (RUN_PHASE == "pilot") and PILOT_EARLY_STOPPING
    EARLY_STOPPING_MONITOR = "val_loss"
    PATIENCE = 15
    EVALUATE_TEST = RUN_PHASE == "final"


def float_tag(value):
    return str(value).replace("-", "m").replace(".", "p")


cfg = Config()

# Computed outside the class body: a class-body generator expression cannot see
# other class-level names (PILOT_CONFIGS), so resolve NEEDS_ASSISTANT via cfg here.
_active_sweep = cfg.PILOT_SWEEP if cfg.RUN_PHASE == "pilot" else cfg.FINAL_SWEEP
cfg.NEEDS_ASSISTANT = any(cfg.PILOT_CONFIGS[p]["use_kd"] for p in _active_sweep)


def build_pilot_setup_id(preset, use_kd, temperature, alpha, stage2_lr, assistant_img_size):
    mode = "ta-kd" if use_kd else "ce-finetune"
    kd_tag = f"t{float_tag(temperature)}_a{float_tag(alpha)}_av{assistant_img_size}" if use_kd else "no-kd"
    return (
        f"r8_k5_pilot_wn256_{mode}_{preset}_s{cfg.SEED}_"
        f"{kd_tag}_lr{float_tag(stage2_lr)}_e{cfg.PILOT_EPOCHS}_"
        f"img{cfg.STUDENT_IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )


def build_final_setup_id(preset, use_kd, temperature, alpha, stage2_lr, assistant_img_size):
    mode = "ta-kd" if use_kd else "ce-finetune"
    kd_tag = f"t{float_tag(temperature)}_a{float_tag(alpha)}_av{assistant_img_size}" if use_kd else "no-kd"
    return (
        f"r8_k5_final_wn256_{mode}_{preset}_s{cfg.SEED}_"
        f"{kd_tag}_lr{float_tag(stage2_lr)}_e{cfg.FINAL_EPOCHS}_"
        f"img{cfg.STUDENT_IMG_SIZE}_bs{cfg.BATCH_SIZE}"
    )


VALID_FINAL_SEEDS = {42, 123, 777, 2026, 3407}
if cfg.RUN_PHASE not in {"pilot", "final"}:
    raise ValueError(f"Unsupported RUN_PHASE: {cfg.RUN_PHASE}")
if cfg.STUDENT_IMG_SIZE != 160:
    raise ValueError("R8-K5 must keep STUDENT_IMG_SIZE=160 to match K6 R8 WasteNet-256K.")
if cfg.ASSISTANT_IMG_SIZE != 380:
    raise ValueError("R8-K5 must keep ASSISTANT_IMG_SIZE=380 to match Focus-RCNet R2-K5.")
if cfg.RUN_PHASE == "pilot":
    if cfg.SEED != 42:
        raise ValueError("R8-K5 pilot sweep uses seed 42 only.")
    if not cfg.PILOT_SWEEP:
        raise ValueError("PILOT_SWEEP is empty.")
    for _preset in cfg.PILOT_SWEEP:
        if _preset not in cfg.PILOT_CONFIGS:
            raise ValueError(f"Unknown preset in PILOT_SWEEP: {_preset}. Choose from {list(cfg.PILOT_CONFIGS)}")
if cfg.RUN_PHASE == "final":
    if cfg.SEED not in VALID_FINAL_SEEDS:
        raise ValueError(f"Final seed must be one of {sorted(VALID_FINAL_SEEDS)}.")
    if cfg.EPOCHS != 100:
        raise ValueError("R8-K5 final protocol requires exactly 100 epochs.")
    if cfg.EARLY_STOPPING:
        raise ValueError("Early stopping must be disabled for final runs.")
    if not cfg.FINAL_SWEEP:
        raise ValueError("FINAL_SWEEP is empty. Freeze the final config(s) from the diagnostic pilot first.")
    for _preset in cfg.FINAL_SWEEP:
        if _preset not in cfg.PILOT_CONFIGS:
            raise ValueError(f"Unknown preset in FINAL_SWEEP: {_preset}. Choose from {list(cfg.PILOT_CONFIGS)}")
if cfg.EVALUATE_TEST != (cfg.RUN_PHASE == "final"):
    raise ValueError("Test evaluation must be disabled for pilot and enabled for final.")

cfg.TRAINING_MODE = "ta_kd" if cfg.USE_KD else "ce_finetune"
cfg.KD_TYPE = "logits_teacher_assistant" if cfg.USE_KD else "ce_finetune_control"

# Per-config SETUP_ID / OUTPUT_DIR are assigned inside the sweep/final driver.
cfg.SETUP_ID = None
cfg.OUTPUT_DIR = None

print(f"Experiment : {cfg.EXPERIMENT_NAME}")
print(f"Run phase  : {cfg.RUN_PHASE}")
print(f"Seed       : {cfg.SEED}")
print(f"Epochs     : {cfg.EPOCHS}")
print(f"Student img: {cfg.STUDENT_IMG_SIZE} | Assistant img: {cfg.ASSISTANT_IMG_SIZE}")
print(f"Early stop : {cfg.EARLY_STOPPING} (patience={cfg.PATIENCE})")
print(f"Eval test  : {cfg.EVALUATE_TEST}")
print(f"Needs assistant: {cfg.NEEDS_ASSISTANT}")
print(f"Dataset dir: {cfg.DATASET_DIR}")
if cfg.RUN_PHASE == "pilot":
    print(f"Pilot sweep ({len(cfg.PILOT_SWEEP)} configs, one Run All):")
    for _p in cfg.PILOT_SWEEP:
        _c = cfg.PILOT_CONFIGS[_p]
        print(f"  - {_p}: use_kd={_c['use_kd']} T={_c['temperature']} alpha={_c['alpha']} lr={_c['stage2_lr']}")
else:
    print(f"Final sweep ({len(cfg.FINAL_SWEEP)} configs, one Run All per seed, test eval ON):")
    for _p in cfg.FINAL_SWEEP:
        _c = cfg.PILOT_CONFIGS[_p]
        print(f"  - {_p}: use_kd={_c['use_kd']} T={_c['temperature']} alpha={_c['alpha']} lr={_c['stage2_lr']} av={_c.get('assistant_img_size', cfg.ASSISTANT_IMG_SIZE)}")

In [ ]:
# ============================================================
# 3. Reproducibility and Device
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = cfg.USE_AMP and device.type == "cuda"
print(f"Device: {device}")
print(f"AMP enabled: {AMP_ENABLED}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"VRAM GB: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

## 1. Load R0-K5 Artifacts

Verifikasi `dataset_name=TrashNet-K5`, `dataset_version=2`, `num_classes=5`, `trash` absen, lalu resolve R2-K5 assistant dan R6-K5 student-init pada seed yang sama.


In [ ]:
# ============================================================
# 4. Resolve R0-K5 Artifacts, R2-K5 Assistant, and R6-K5 Student Init
# ============================================================

def has_r0_k5_artifacts(path: Path) -> bool:
    mapping_path = path / "class_mapping.json"
    if not mapping_path.exists() or not any(path.glob("split_manifest_seed_*.csv")):
        return False
    try:
        mapping = json.loads(mapping_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        mapping.get("dataset_name") == cfg.DATASET_NAME
        and mapping.get("dataset_version") == cfg.DATASET_VERSION
        and mapping.get("num_classes") == 5
        and "trash" not in mapping.get("class_names", [])
    )


def candidate_variants(candidate: Path):
    yield candidate
    yield candidate / "r0_data_protocol"
    yield candidate / "final_research_kd_k5" / "r0_data_protocol"


def discover_r0_dirs():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    discovered = []
    for root in roots:
        if not root.exists():
            continue
        try:
            discovered.extend([path for path in root.rglob("r0_data_protocol") if path.is_dir()])
        except Exception as exc:
            print(f"Skipping R0 discovery under {root}: {exc}")
    return discovered


def resolve_existing_dir(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if str(candidate) in {"", "."}:
            continue
        for variant in candidate_variants(candidate):
            if variant.exists() and has_r0_k5_artifacts(variant):
                return variant

    discovered = discover_r0_dirs()
    valid_discovered = [path for path in discovered if has_r0_k5_artifacts(path)]
    if valid_discovered:
        print("Auto-discovered R0 candidates:")
        for path in valid_discovered:
            print(f"- {path}")
        return valid_discovered[0]

    print("Checked R0 candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    print("Discovered r0_data_protocol dirs:")
    for path in discovered:
        print(f"- {path}")
    raise FileNotFoundError("R0-K5 data protocol directory not found. Set R0_K5_DATA_PROTOCOL_DIR.")


def resolve_checkpoint(seed, roots, filename, label, extra_subdirs=None):
    extra_subdirs = extra_subdirs or []
    candidates = []
    for root in roots:
        root = Path(root)
        if str(root) in {"", "."}:
            continue
        candidates.extend([
            root / f"seed_{seed}" / filename,
            root / filename,
        ])
        for subdir in extra_subdirs:
            subdir = Path(subdir)
            candidates.extend([
                root / subdir / f"seed_{seed}" / filename,
                root / "final_research_kd_k5" / "runs" / "final" / subdir / f"seed_{seed}" / filename,
                root / "runs" / "final" / subdir / f"seed_{seed}" / filename,
            ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            try:
                matches = list(root.rglob(filename))
            except Exception as exc:
                print(f"Skipping checkpoint discovery under {root}: {exc}")
                matches = []
            if matches:
                return matches[0]
    print(f"Checked {label} checkpoint candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    raise FileNotFoundError(f"{label} checkpoint not found for seed {seed}. Attach output or set the matching env var.")


def resolve_assistant_checkpoint(seed: int) -> Path:
    filename = f"focus_rcnet_ce_r2_final_seed_{seed}_best.pth"
    return resolve_checkpoint(seed, cfg.R2_ASSISTANT_ROOT_CANDIDATES, filename, "R2-K5 Focus-RCNet assistant", ["R2", Path("final") / "R2"])


def resolve_student_init_checkpoint(seed: int) -> Path:
    filename = f"wastenet_256k_ce_baseline_r6_final_seed_{seed}_best.pth"
    return resolve_checkpoint(seed, cfg.R6_STUDENT_ROOT_CANDIDATES, filename, "R6-K5 WasteNet-256K CE student init", ["R6", Path("final") / "R3"])


R0_DIR = resolve_existing_dir(cfg.R0_DIR_CANDIDATES)
CLASS_MAPPING_PATH = R0_DIR / "class_mapping.json"
MANIFEST_PATH = R0_DIR / f"split_manifest_seed_{cfg.SEED}.csv"
EXCLUDED_DUPLICATES_PATH = R0_DIR / "excluded_duplicate_conflicts.csv"
STUDENT_INIT_CHECKPOINT_PATH = resolve_student_init_checkpoint(cfg.SEED)
ASSISTANT_CHECKPOINT_PATH = resolve_assistant_checkpoint(cfg.SEED) if cfg.NEEDS_ASSISTANT else None

required_paths = [CLASS_MAPPING_PATH, MANIFEST_PATH, EXCLUDED_DUPLICATES_PATH, STUDENT_INIT_CHECKPOINT_PATH]
if ASSISTANT_CHECKPOINT_PATH is not None:
    required_paths.append(ASSISTANT_CHECKPOINT_PATH)
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path}")

with CLASS_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

if class_mapping.get("dataset_name") != cfg.DATASET_NAME:
    raise ValueError(f"Wrong R0 dataset: {class_mapping.get('dataset_name')}")
if class_mapping.get("dataset_version") != cfg.DATASET_VERSION:
    raise ValueError(f"Wrong R0 dataset version: {class_mapping.get('dataset_version')}")
if class_mapping.get("num_classes") != 5 or "trash" in class_mapping.get("class_names", []):
    raise ValueError("R8-K5 requires exactly five non-trash classes.")

manifest_df = pd.read_csv(MANIFEST_PATH)
excluded_df = pd.read_csv(EXCLUDED_DUPLICATES_PATH)
if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

CLASS_NAMES = class_mapping["class_names"]
CLASS_TO_IDX = class_mapping["class_to_idx"]
NUM_CLASSES = len(CLASS_NAMES)

print(f"R0 dir          : {R0_DIR}")
print(f"Manifest        : {MANIFEST_PATH.name}")
print(f"Student init    : {STUDENT_INIT_CHECKPOINT_PATH}")
print(f"Assistant ckpt  : {ASSISTANT_CHECKPOINT_PATH if ASSISTANT_CHECKPOINT_PATH else 'not required (no KD config in this run)'}")
print(f"Rows            : {len(manifest_df)}")
print(f"Classes         : {CLASS_NAMES}")
print(f"Excluded dup    : {len(excluded_df)} rows")
display(manifest_df.groupby(["split", "label", "class_id"], as_index=False).size())

In [ ]:
# ============================================================
# 5. Manifest Validation
# ============================================================

required_columns = {"sample_id", "image_path", "relative_path", "label", "class_id", "split", "seed", "sha256"}
missing_columns = required_columns - set(manifest_df.columns)
if missing_columns:
    raise ValueError(f"Manifest missing columns: {sorted(missing_columns)}")

if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

if set(manifest_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Manifest must contain train, val, and test splits.")

excluded_ids = set(excluded_df.get("sample_id", []))
if excluded_ids & set(manifest_df["sample_id"]):
    raise AssertionError("Excluded duplicate-conflict samples are present in this manifest.")

for class_name, class_id in CLASS_TO_IDX.items():
    rows = manifest_df[manifest_df["label"] == class_name]
    if rows.empty:
        raise AssertionError(f"Missing class in manifest: {class_name}")
    if set(rows["class_id"].unique()) != {class_id}:
        raise AssertionError(f"Class id mismatch for {class_name}")

print("Manifest validation passed")

## 2. Dataset and DataLoader


In [ ]:
# ============================================================
# 6. Dual-Resolution Transforms
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def coarse_dropout(img_size: int):
    min_h = int(img_size * 0.05)
    max_h = int(img_size * 0.20)
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 1),
            hole_height_range=(min_h, max_h),
            hole_width_range=(min_h, max_h),
            fill=0,
            p=0.5,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=1,
            min_height=min_h,
            max_height=max_h,
            min_width=min_h,
            max_width=max_h,
            fill_value=0,
            p=0.5,
        )


def shared_train_transform(img_size: int):
    # One shared random augmentation applied before both model views are created.
    # Plain A.Compose -> uses the numpy global RNG, which seed_everything resets per
    # config, so the augmentation order is identical across sweep configs.
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        coarse_dropout(img_size),
    ])


def model_view_transform(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


train_shared_transform = shared_train_transform(cfg.ASSISTANT_IMG_SIZE)
student_view_transform = model_view_transform(cfg.STUDENT_IMG_SIZE)
assistant_view_transform = model_view_transform(cfg.ASSISTANT_IMG_SIZE)
student_eval_transform = model_view_transform(cfg.STUDENT_IMG_SIZE)

print("Paired dual-resolution transforms ready")
print("One shared random augmentation is applied before both model views are created.")
print(f"Student image   : {cfg.STUDENT_IMG_SIZE}x{cfg.STUDENT_IMG_SIZE}")
print(f"Assistant image : {cfg.ASSISTANT_IMG_SIZE}x{cfg.ASSISTANT_IMG_SIZE}")

In [ ]:
# ============================================================
# 7. Dual-Resolution Manifest Dataset
# ============================================================

class DualResolutionTrashNetDataset(Dataset):
    def __init__(self, df, dataset_dir, shared_transform=None, student_transform=None, assistant_transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.dataset_dir = Path(dataset_dir)
        self.shared_transform = shared_transform
        self.student_transform = student_transform
        self.assistant_transform = assistant_transform

    def __len__(self):
        return len(self.df)

    def resolve_path(self, row) -> Path:
        absolute_path = Path(row["image_path"])
        if absolute_path.exists():
            return absolute_path
        fallback_path = self.dataset_dir / row["relative_path"]
        if fallback_path.exists():
            return fallback_path
        raise FileNotFoundError(f"Image not found: {absolute_path} or {fallback_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.resolve_path(row)
        image = np.array(Image.open(image_path).convert("RGB"))
        paired_image = self.shared_transform(image=image)["image"] if self.shared_transform else image
        student_image = self.student_transform(image=paired_image)["image"] if self.student_transform else paired_image
        assistant_image = self.assistant_transform(image=paired_image)["image"] if self.assistant_transform else student_image
        return {
            "student_image": student_image,
            "assistant_image": assistant_image,
            "label": int(row["class_id"]),
            "sample_id": row["sample_id"],
            "relative_path": row["relative_path"],
        }


train_df = manifest_df[manifest_df["split"] == "train"].copy()
val_df = manifest_df[manifest_df["split"] == "val"].copy()
test_df = manifest_df[manifest_df["split"] == "test"].copy()

# Assistant view only needed during training (KD). Eval reads the student view only.
train_dataset = DualResolutionTrashNetDataset(
    train_df,
    cfg.DATASET_DIR,
    shared_transform=train_shared_transform,
    student_transform=student_view_transform,
    assistant_transform=assistant_view_transform if cfg.NEEDS_ASSISTANT else None,
)
val_dataset = DualResolutionTrashNetDataset(val_df, cfg.DATASET_DIR, student_transform=student_eval_transform)
test_dataset = DualResolutionTrashNetDataset(test_df, cfg.DATASET_DIR, student_transform=student_eval_transform)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# ============================================================
# 8. DataLoaders and Distribution
# ============================================================

def worker_init_fn(worker_id):
    worker_seed = cfg.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(cfg.SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)

split_counts = manifest_df.groupby(["split", "label", "class_id"], as_index=False).size()
display(split_counts)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

## 3. Student, Assistant, and Training Components


In [ ]:
# ============================================================
# 9. WasteNet-256K Student and Focus-RCNet Assistant
# ============================================================

@dataclass(frozen=True)
class WasteNetVariant:
    variant_id: str
    display_name: str
    channels: tuple
    repeats: tuple
    expand_ratio: float
    classifier_hidden: int
    expected_params: int
    checkpoint_prefix: str


WASTENET_256K = WasteNetVariant(
    variant_id="wastenet_256k",
    display_name="WasteNet-256K",
    channels=(16, 32, 72, 128, 224),
    repeats=(1, 1, 2, 1),
    expand_ratio=2.5,
    classifier_hidden=0,
    expected_params=255_777,
    checkpoint_prefix="wastenet_256k",
)


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, groups: int = 1):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: float = 1.0):
        super().__init__()
        hidden_channels = int(round(in_channels * expand_ratio))
        layers = []
        if hidden_channels != in_channels:
            layers.append(ConvBNAct(in_channels, hidden_channels, kernel_size=1, stride=1))
        layers.extend([
            ConvBNAct(hidden_channels, hidden_channels, kernel_size=3, stride=stride, groups=hidden_channels),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.block = nn.Sequential(*layers)
        self.activation = nn.SiLU(inplace=True)
        self.use_residual = stride == 1 and in_channels == out_channels

    def forward(self, x):
        y = self.block(x)
        if self.use_residual:
            y = y + x
        return self.activation(y)


class WasteNet(nn.Module):
    def __init__(self, variant: WasteNetVariant, num_classes: int = 5, dropout: float = 0.2):
        super().__init__()
        layers = [ConvBNAct(3, variant.channels[0], kernel_size=3, stride=2)]
        in_channels = variant.channels[0]

        for out_channels, repeat in zip(variant.channels[1:], variant.repeats):
            for block_idx in range(repeat):
                stride = 2 if block_idx == 0 else 1
                layers.append(
                    DepthwiseSeparableBlock(
                        in_channels,
                        out_channels,
                        stride=stride,
                        expand_ratio=variant.expand_ratio,
                    )
                )
                in_channels = out_channels

        self.features = nn.Sequential(*layers)
        if variant.classifier_hidden > 0:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, variant.classifier_hidden),
                nn.SiLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(variant.classifier_hidden, num_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, num_classes),
            )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def create_wastenet_256k() -> WasteNet:
    return WasteNet(variant=WASTENET_256K, num_classes=NUM_CLASSES)


# ============================================================
# Focus-RCNet Assistant (identical architecture to R2-K5)
# ============================================================


class Focus(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels * 4, out_channels, kernel_size, stride=1, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(torch.cat([
            x[..., ::2, ::2],
            x[..., 1::2, ::2],
            x[..., ::2, 1::2],
            x[..., 1::2, 1::2],
        ], dim=1))


class SimAM(nn.Module):
    def __init__(self, e_lambda: float = 1e-4):
        super().__init__()
        self.e_lambda = e_lambda

    def forward(self, x):
        _, _, height, width = x.size()
        n = height * width - 1
        x_minus_mu_sq = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_sq / (
            4 * (x_minus_mu_sq.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * torch.sigmoid(y)


class SandglassBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 2):
        super().__init__()
        self.use_residual = stride == 1 and in_channels == out_channels
        mid_channels = max(in_channels // reduction, 1)
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.Conv2d(mid_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        output = self.layers(x)
        return x + output if self.use_residual else output


class FocusRCNet(nn.Module):
    STAGE_CONFIG = [(48, 4, 2, 2), (96, 3, 2, 2), (192, 2, 2, 2), (384, 2, 2, 2)]

    def __init__(self, num_classes: int = 5, dropout: float = 0.2):
        super().__init__()
        self.focus = Focus(3, 24, kernel_size=1)
        stages = []
        in_channels = 24
        for out_channels, num_blocks, stride, reduction in self.STAGE_CONFIG:
            blocks = []
            for block_idx in range(num_blocks):
                block_stride = stride if block_idx == 0 else 1
                blocks.append(SandglassBlock(in_channels, out_channels, stride=block_stride, reduction=reduction))
                in_channels = out_channels
            blocks.append(SimAM())
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes),
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.focus(x)
        x = self.stages(x)
        x = self.conv5(x)
        return self.classifier(x)


def create_focus_rcnet(num_classes: int = 5):
    return FocusRCNet(num_classes=num_classes)


def load_student_initialization(model: nn.Module, checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(loaded["model_state_dict"])
    print(f"Student init loaded: {checkpoint_path}")
    print(f"R6-K5 best epoch   : {loaded.get('best_epoch', 'n/a')}")
    print(f"R6-K5 best val acc : {loaded.get('best_val_acc', 'n/a')}")
    return loaded


def load_assistant_model(checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assistant = create_focus_rcnet(NUM_CLASSES)
    assistant.load_state_dict(loaded["model_state_dict"])
    assistant = assistant.to(device).eval()
    for parameter in assistant.parameters():
        parameter.requires_grad = False
    print(f"Assistant loaded   : {checkpoint_path}")
    print(f"R2-K5 best epoch   : {loaded.get('best_epoch', 'n/a')}")
    print(f"R2-K5 best val acc : {loaded.get('best_val_acc', 'n/a')}")
    return assistant, loaded


# Param sanity check on a throwaway student, plus load the (shared) assistant once.
_probe = create_wastenet_256k().to(device)
total_params = sum(p.numel() for p in _probe.parameters())
trainable_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)

sample = torch.randn(1, 3, cfg.STUDENT_IMG_SIZE, cfg.STUDENT_IMG_SIZE).to(device)
with torch.no_grad():
    sample_output = _probe(sample)
assert tuple(sample_output.shape) == (1, NUM_CLASSES), f"Unexpected output shape: {sample_output.shape}"
assert total_params == WASTENET_256K.expected_params, f"Expected 255,777 params, got {total_params:,}"
del _probe, sample, sample_output

assistant_model = None
assistant_checkpoint = None
if cfg.NEEDS_ASSISTANT:
    assistant_model, assistant_checkpoint = load_assistant_model(ASSISTANT_CHECKPOINT_PATH)
    assistant_sample = torch.randn(1, 3, cfg.ASSISTANT_IMG_SIZE, cfg.ASSISTANT_IMG_SIZE).to(device)
    with torch.no_grad():
        assistant_output = assistant_model(assistant_sample)
    assert tuple(assistant_output.shape) == (1, NUM_CLASSES), f"Unexpected assistant output shape: {assistant_output.shape}"
    del assistant_sample, assistant_output

print(f"Student model : {cfg.MODEL_NAME}")
print(f"Assistant     : {cfg.ASSISTANT_MODEL_NAME if cfg.NEEDS_ASSISTANT else 'not used (no KD config in this run)'}")
print(f"Variant       : {WASTENET_256K.variant_id}")
print(f"Total params  : {total_params:,}")
print(f"Trainable     : {trainable_params:,}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 10. Loss, KD, and Train / Validate Helpers
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = None  # assigned per config in the driver
scaler = GradScaler(enabled=AMP_ENABLED)


def kd_loss(student_logits, assistant_logits, labels):
    temperature = cfg.KD_TEMPERATURE
    alpha = cfg.KD_ALPHA
    student_log_soft = F.log_softmax(student_logits.float() / temperature, dim=1)
    assistant_soft = F.softmax(assistant_logits.float() / temperature, dim=1)
    loss_soft = F.kl_div(student_log_soft, assistant_soft, reduction="batchmean") * (temperature ** 2)
    loss_hard = criterion(student_logits.float(), labels)
    loss = alpha * loss_soft + (1.0 - alpha) * loss_hard
    return loss, loss_soft, loss_hard


def run_one_train_epoch(student, assistant, loader, use_kd):
    student.train()
    if assistant is not None:
        assistant.eval()
    total_loss = 0.0
    total_soft_loss = 0.0
    total_hard_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        student_images = batch["student_image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=AMP_ENABLED):
            student_logits = student(student_images)
            if use_kd:
                assistant_images = batch["assistant_image"].to(device, non_blocking=True)
                with torch.no_grad():
                    assistant_logits = assistant(assistant_images)
        with autocast(enabled=False):
            if use_kd:
                loss, loss_soft, loss_hard = kd_loss(student_logits, assistant_logits, labels)
                soft_value = loss_soft.item()
                hard_value = loss_hard.item()
            else:
                loss = criterion(student_logits.float(), labels)
                soft_value = float("nan")
                hard_value = loss.item()
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss detected")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_soft_loss += soft_value * batch_size
        total_hard_loss += hard_value * batch_size
        total_correct += (student_logits.argmax(dim=1) == labels).sum().item()
        total += batch_size
    return total_loss / total, total_correct / total, total_soft_loss / total, total_hard_loss / total


@torch.no_grad()
def run_eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        images = batch["student_image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, total_correct / total

print("Training components ready")

In [ ]:
# ============================================================
# 11. Evaluation Helpers
# ============================================================

@torch.no_grad()
def collect_predictions(model, loader, split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in loader:
        images = batch["student_image"].to(device, non_blocking=True)
        labels = batch["label"].cpu().numpy()
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        for i in range(len(labels)):
            row = {
                "sample_id": batch["sample_id"][i],
                "image_path": batch["relative_path"][i],
                "label": CLASS_NAMES[int(labels[i])],
                "label_id": int(labels[i]),
                "prediction": CLASS_NAMES[int(preds[i])],
                "prediction_id": int(preds[i]),
                "seed": cfg.SEED,
                "model_id": cfg.EXPERIMENT_ID,
                "variant": cfg.TRAINING_MODE,
                "split": split_name,
            }
            for class_idx, class_name in enumerate(CLASS_NAMES):
                row[f"prob_{class_name}"] = float(probs[i, class_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label_id"].to_numpy()
    y_pred = pred_df["prediction_id"].to_numpy()
    prob_cols = [f"prob_{class_name}" for class_name in CLASS_NAMES]
    y_prob = pred_df[prob_cols].to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    try:
        auc_macro_ovr = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc_macro_ovr = np.nan

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "auc_macro_ovr": float(auc_macro_ovr),
    }


def save_confusion_matrix(pred_df: pd.DataFrame, split_name: str):
    cm = confusion_matrix(pred_df["label_id"], pred_df["prediction_id"], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{cfg.EXPERIMENT_ID} WasteNet-256K TA-KD - {split_name}")
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    path = cfg.OUTPUT_DIR / f"confusion_matrix_{split_name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path

## 4. Train / Evaluate Driver


In [ ]:
# ============================================================
# 12-17. TA-KD train / eval / save driver (one config)
# ============================================================

def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_one_kd_config(use_kd, temperature, alpha, stage2_lr, assistant_img_size, setup_id, output_dir, preset=None):
    """Train + evaluate + save WasteNet-256K for one R8 config. Returns a val summary dict."""
    global optimizer, scaler

    cfg.USE_KD = use_kd
    cfg.KD_TEMPERATURE = temperature
    cfg.KD_ALPHA = alpha
    cfg.STAGE2_LR = stage2_lr
    cfg.ASSISTANT_IMG_SIZE_USED = assistant_img_size
    cfg.TRAINING_MODE = "ta_kd" if use_kd else "ce_finetune"
    cfg.KD_TYPE = "logits_teacher_assistant" if use_kd else "ce_finetune_control"
    cfg.SETUP_ID = setup_id
    cfg.OUTPUT_DIR = Path(output_dir)
    cfg.PILOT_PRESET = preset
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    assistant = assistant_model if use_kd else None
    if use_kd and assistant is None:
        raise RuntimeError("KD config requested but assistant model is not loaded (cfg.NEEDS_ASSISTANT).")

    # Reproducible: fresh student initialised from R6-K5, identical data order, so
    # configs differ only by (use_kd, T, alpha, stage2_lr, assistant_img_size).
    seed_everything(cfg.SEED)
    generator.manual_seed(cfg.SEED)

    # Build the train loader at this config's assistant resolution (student stays 160).
    # The shared aug runs at ASSISTANT_IMG_SIZE; the assistant view downscales to its size,
    # so av160 == assistant sees the same 160px the student does (matched resolution).
    config_assistant_transform = model_view_transform(assistant_img_size) if use_kd else None
    config_train_dataset = DualResolutionTrashNetDataset(
        train_df,
        cfg.DATASET_DIR,
        shared_transform=train_shared_transform,
        student_transform=student_view_transform,
        assistant_transform=config_assistant_transform,
    )
    config_train_loader = DataLoader(
        config_train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
        generator=generator,
    )

    model = create_wastenet_256k().to(device)
    load_student_initialization(model, STUDENT_INIT_CHECKPOINT_PATH)
    optimizer = optim.SGD(model.parameters(), lr=stage2_lr, momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    scaler = GradScaler(enabled=AMP_ENABLED)

    history = []
    best_val_acc = -1.0
    best_epoch = 0
    best_model_state = None
    best_monitor_value = np.inf if cfg.EARLY_STOPPING_MONITOR == "val_loss" else -np.inf
    epochs_without_improvement = 0
    total_start = time.time()

    print(f"Starting {cfg.EXPERIMENT_NAME} | seed={cfg.SEED} | mode={cfg.TRAINING_MODE} | T={temperature} | alpha={alpha} | lr={stage2_lr} | assistant_img={assistant_img_size if use_kd else 'n/a'}")
    for epoch in range(1, cfg.EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss, train_acc, train_soft_loss, train_hard_loss = run_one_train_epoch(model, assistant, config_train_loader, use_kd)
        val_loss, val_acc = run_eval_epoch(model, val_loader)
        scheduler.step()
        epoch_time = time.time() - epoch_start
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
        if cfg.EARLY_STOPPING_MONITOR == "val_loss":
            monitor_value = val_loss
            monitor_improved = monitor_value < best_monitor_value - 1e-8
        elif cfg.EARLY_STOPPING_MONITOR == "val_acc":
            monitor_value = val_acc
            monitor_improved = monitor_value > best_monitor_value + 1e-8
        else:
            raise ValueError(f"Unsupported EARLY_STOPPING_MONITOR: {cfg.EARLY_STOPPING_MONITOR}")
        if monitor_improved:
            best_monitor_value = monitor_value
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_loss_soft": train_soft_loss,
            "train_loss_hard": train_hard_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "epoch_time_sec": epoch_time,
            "is_best": is_best,
            "early_stop_monitor": cfg.EARLY_STOPPING_MONITOR,
            "early_stop_value": monitor_value,
            "early_stop_improved": monitor_improved,
            "epochs_without_improvement": epochs_without_improvement,
        })
        marker = " BEST" if is_best else ""
        early_stop_status = f" | es_wait={epochs_without_improvement}/{cfg.PATIENCE}" if cfg.EARLY_STOPPING else ""
        print(
            f"Epoch {epoch:03d}/{cfg.EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"soft={train_soft_loss:.4f} hard={train_hard_loss:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"lr={current_lr:.6f} time={epoch_time:.1f}s{early_stop_status}{marker}"
        )
        if cfg.EARLY_STOPPING and epochs_without_improvement >= cfg.PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} ({cfg.EARLY_STOPPING_MONITOR} did not improve for {cfg.PATIENCE} epochs).")
            break

    total_train_time = time.time() - total_start
    history_df = pd.DataFrame(history)
    if best_model_state is None:
        raise RuntimeError("No best model state was captured.")
    print("Training complete")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val acc: {best_val_acc:.6f}")
    print(f"Total minutes: {total_train_time / 60:.1f}")

    model.load_state_dict(best_model_state)

    val_predictions_df = collect_predictions(model, val_loader, "val")
    val_metrics = compute_metrics(val_predictions_df)

    test_predictions_df = pd.DataFrame()
    test_metrics = None
    if cfg.EVALUATE_TEST and cfg.RUN_PHASE == "final":
        test_predictions_df = collect_predictions(model, test_loader, "test")
        test_metrics = compute_metrics(test_predictions_df)
    else:
        print("Independent test evaluation skipped. Set RUN_PHASE='final' to enable it.")

    print("Validation metrics:")
    display(pd.DataFrame([val_metrics]))
    if test_metrics is not None:
        print("Test metrics:")
        display(pd.DataFrame([test_metrics]))

    prefix = f"{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}"

    val_pred_path = cfg.OUTPUT_DIR / f"predictions_{prefix}_val.csv"
    val_predictions_df.to_csv(val_pred_path, index=False)
    print(f"Saved: {val_pred_path}")

    test_pred_path = None
    if not test_predictions_df.empty:
        test_pred_path = cfg.OUTPUT_DIR / f"predictions_{prefix}_test.csv"
        test_predictions_df.to_csv(test_pred_path, index=False)
        print(f"Saved: {test_pred_path}")

    val_cm_path = save_confusion_matrix(val_predictions_df, "val")
    test_cm_path = None
    if not test_predictions_df.empty:
        test_cm_path = save_confusion_matrix(test_predictions_df, "test")

    history_path = cfg.OUTPUT_DIR / f"training_history_{prefix}.csv"
    history_df.to_csv(history_path, index=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
    axes[0].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(history_df["epoch"], history_df["train_acc"], label="train")
    axes[1].plot(history_df["epoch"], history_df["val_acc"], label="val")
    axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    axes[2].plot(history_df["epoch"], history_df["lr"], color="green", label="lr")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)
    fig.suptitle(f"{cfg.EXPERIMENT_ID} WasteNet-256K R8-K5 TA-KD - seed {cfg.SEED} | mode={cfg.TRAINING_MODE}, T={temperature}, alpha={alpha}, lr={stage2_lr}")
    fig.tight_layout()
    curve_path = cfg.OUTPUT_DIR / f"training_curves_{prefix}.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {history_path}")
    print(f"Saved: {curve_path}")

    metrics_rows = [{"split": "val", **val_metrics}]
    if test_metrics is not None:
        metrics_rows.append({"split": "test", **test_metrics})
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_path = cfg.OUTPUT_DIR / f"metrics_{prefix}.csv"
    metrics_df.to_csv(metrics_path, index=False)

    config_dict = {
        "experiment_id": cfg.EXPERIMENT_ID,
        "experiment_name": cfg.EXPERIMENT_NAME,
        "dataset_name": cfg.DATASET_NAME,
        "dataset_version": cfg.DATASET_VERSION,
        "protocol": "R0-K5 stratified 70/15/15",
        "seed": cfg.SEED,
        "run_phase": cfg.RUN_PHASE,
        "setup_id": cfg.SETUP_ID,
        "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
        "output_root": str(cfg.OUTPUT_ROOT),
        "output_dir": str(cfg.OUTPUT_DIR),
        "dataset_dir": str(cfg.DATASET_DIR),
        "r0_dir": str(R0_DIR),
        "manifest_path": str(MANIFEST_PATH),
        "split_manifest_sha256": file_sha256(MANIFEST_PATH),
        "student_init_checkpoint_path": str(STUDENT_INIT_CHECKPOINT_PATH),
        "student_init_checkpoint_sha256": file_sha256(STUDENT_INIT_CHECKPOINT_PATH),
        "assistant_checkpoint_path": str(ASSISTANT_CHECKPOINT_PATH) if (use_kd and ASSISTANT_CHECKPOINT_PATH) else None,
        "assistant_checkpoint_sha256": file_sha256(ASSISTANT_CHECKPOINT_PATH) if (use_kd and ASSISTANT_CHECKPOINT_PATH) else None,
        "assistant_model_name": cfg.ASSISTANT_MODEL_NAME,
        "assistant_source_variant": cfg.ASSISTANT_SOURCE_VARIANT,
        "model_name": cfg.MODEL_NAME,
        "variant_id": WASTENET_256K.variant_id,
        "variant": {
            "channels": WASTENET_256K.channels,
            "repeats": WASTENET_256K.repeats,
            "expand_ratio": WASTENET_256K.expand_ratio,
            "classifier_hidden": WASTENET_256K.classifier_hidden,
            "expected_params": WASTENET_256K.expected_params,
        },
        "training_mode": cfg.TRAINING_MODE,
        "knowledge_distillation": bool(use_kd),
        "kd_type": cfg.KD_TYPE,
        "kd_temperature": temperature,
        "kd_alpha": alpha,
        "stage2_lr": stage2_lr,
        "pilot_grid_temperatures": cfg.PILOT_GRID_TEMPERATURES,
        "pilot_grid_alphas": cfg.PILOT_GRID_ALPHAS,
        "pilot_stage2_lrs": cfg.PILOT_STAGE2_LRS,
        "pilot_anchor": cfg.PILOT_ANCHOR,
        "pilot_configs": cfg.PILOT_CONFIGS,
        "pilot_sweep": cfg.PILOT_SWEEP,
        "student_img_size": cfg.STUDENT_IMG_SIZE,
        "assistant_img_size": assistant_img_size,
        "epochs": cfg.EPOCHS,
        "final_epochs": cfg.FINAL_EPOCHS,
        "pilot_epochs": cfg.PILOT_EPOCHS,
        "batch_size": cfg.BATCH_SIZE,
        "optimizer": "SGD",
        "lr": stage2_lr,
        "momentum": cfg.MOMENTUM,
        "weight_decay": cfg.WEIGHT_DECAY,
        "scheduler": cfg.SCHEDULER,
        "use_amp": AMP_ENABLED,
        "paired_augmentation": True,
        "paired_augmentation_base_size": cfg.ASSISTANT_IMG_SIZE,
        "early_stopping": cfg.EARLY_STOPPING,
        "evaluate_test": cfg.EVALUATE_TEST,
        "early_stopping_monitor": cfg.EARLY_STOPPING_MONITOR,
        "patience": cfg.PATIENCE,
        "checkpoint_metric": cfg.CHECKPOINT_METRIC,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "train_size": int(len(train_df)),
        "val_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "total_train_time_sec": float(total_train_time),
    }
    checkpoint = {
        "model_state_dict": best_model_state,
        "student_init_checkpoint_path": str(STUDENT_INIT_CHECKPOINT_PATH),
        "assistant_checkpoint_path": str(ASSISTANT_CHECKPOINT_PATH) if (use_kd and ASSISTANT_CHECKPOINT_PATH) else None,
        "best_epoch": best_epoch,
        "best_val_acc": float(best_val_acc),
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "config": config_dict,
        "class_mapping": class_mapping,
    }
    checkpoint_path = cfg.OUTPUT_DIR / f"wastenet_256k_{prefix}_best.pth"
    torch.save(checkpoint, checkpoint_path)
    config_path = cfg.OUTPUT_DIR / f"config_{prefix}.json"
    config_path.write_text(json.dumps(config_dict, indent=2), encoding="utf-8")
    artifact_manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": cfg.EXPERIMENT_ID,
        "dataset_name": cfg.DATASET_NAME,
        "dataset_version": cfg.DATASET_VERSION,
        "seed": cfg.SEED,
        "run_phase": cfg.RUN_PHASE,
        "training_mode": cfg.TRAINING_MODE,
        "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
        "kd_temperature": temperature,
        "kd_alpha": alpha,
        "stage2_lr": stage2_lr,
        "setup_id": cfg.SETUP_ID,
        "output_dir": str(cfg.OUTPUT_DIR),
        "artifacts": {
            "checkpoint": str(checkpoint_path),
            "config": str(config_path),
            "history": str(history_path),
            "metrics": str(metrics_path),
            "val_predictions": str(val_pred_path),
            "test_predictions": str(test_pred_path) if test_pred_path else None,
            "val_confusion_matrix": str(val_cm_path),
            "test_confusion_matrix": str(test_cm_path) if test_cm_path else None,
            "training_curves": str(curve_path),
        },
    }
    artifact_manifest_path = cfg.OUTPUT_DIR / f"artifact_manifest_{prefix}.json"
    artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Saved metrics   : {metrics_path}")
    print(f"Saved manifest  : {artifact_manifest_path}")
    display(metrics_df)

    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    verify_model = create_wastenet_256k()
    verify_model.load_state_dict(loaded["model_state_dict"])
    print("Checkpoint verification passed")
    print(f"Best epoch   : {loaded['best_epoch']}")
    print(f"Best val acc : {loaded['best_val_acc']:.6f}")

    del loaded, verify_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "preset": preset,
        "use_kd": use_kd,
        "temperature": temperature,
        "alpha": alpha,
        "stage2_lr": stage2_lr,
        "assistant_img_size": assistant_img_size,
        "best_epoch": best_epoch,
        "best_val_acc": float(best_val_acc),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_f1_macro": float(val_metrics["f1_macro"]),
        "val_auc": float(val_metrics["auc_macro_ovr"]),
        "test_accuracy": float(test_metrics["accuracy"]) if test_metrics else None,
        "setup_id": setup_id,
        "output_dir": str(cfg.OUTPUT_DIR),
        "checkpoint": str(checkpoint_path),
    }

print("R8-K5 config driver ready")

In [ ]:
# ============================================================
# 18. Run: pilot sweep, or final sweep (both one Run All)
# ============================================================

if cfg.RUN_PHASE == "pilot":
    pilot_root = cfg.OUTPUT_ROOT / "pilots" / cfg.EXPERIMENT_ID
    sweep_results = []
    for preset in cfg.PILOT_SWEEP:
        params = cfg.PILOT_CONFIGS[preset]
        use_kd = params["use_kd"]
        T, alpha, lr = params["temperature"], params["alpha"], params["stage2_lr"]
        av = params.get("assistant_img_size", cfg.ASSISTANT_IMG_SIZE)
        setup_id = build_pilot_setup_id(preset, use_kd, T, alpha, lr, av)
        out_dir = pilot_root / setup_id
        print("\n" + "=" * 78)
        print(f"PILOT CONFIG {len(sweep_results) + 1}/{len(cfg.PILOT_SWEEP)}: {preset} | use_kd={use_kd} T={T} alpha={alpha} lr={lr} assistant_img={av if use_kd else 'n/a'}")
        print("=" * 78)
        sweep_results.append(run_one_kd_config(use_kd, T, alpha, lr, av, setup_id, out_dir, preset=preset))

    sweep_df = pd.DataFrame(sweep_results)[
        ["preset", "use_kd", "temperature", "alpha", "stage2_lr", "assistant_img_size", "best_epoch", "val_accuracy", "val_f1_macro", "val_auc", "setup_id"]
    ].sort_values(["val_accuracy", "val_f1_macro"], ascending=False).reset_index(drop=True)
    pilot_root.mkdir(parents=True, exist_ok=True)
    summary_path = pilot_root / "r8_k5_pilot_sweep_summary.csv"
    sweep_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 78)
    print("PILOT SWEEP SUMMARY (sorted by val accuracy, then macro-F1):")
    display(sweep_df)
    winner = sweep_df.iloc[0]
    print(f"Saved summary: {summary_path}")
    print(
        f"Winner on val: {winner['preset']} (use_kd={winner['use_kd']}, T={winner['temperature']}, "
        f"alpha={winner['alpha']}, lr={winner['stage2_lr']}) "
        f"-> val_acc={winner['val_accuracy']:.4f}, val_f1={winner['val_f1_macro']:.4f}"
    )
    print("Next: put the chosen config(s) in FINAL_SWEEP, RUN_PHASE='final', run 5 seeds.")
    print("Reminder: compare against R6-K5 (no-KD floor) and R7-K5 (direct KD) before declaring an R8 benefit.")
else:
    final_root = cfg.OUTPUT_ROOT / "final" / cfg.EXPERIMENT_ID / f"seed_{cfg.SEED}"
    final_results = []
    for preset in cfg.FINAL_SWEEP:
        params = cfg.PILOT_CONFIGS[preset]
        use_kd = params["use_kd"]
        T, alpha, lr = params["temperature"], params["alpha"], params["stage2_lr"]
        av = params.get("assistant_img_size", cfg.ASSISTANT_IMG_SIZE)
        setup_id = build_final_setup_id(preset, use_kd, T, alpha, lr, av)
        out_dir = final_root / preset
        print("\n" + "=" * 78)
        print(f"FINAL CONFIG {len(final_results) + 1}/{len(cfg.FINAL_SWEEP)}: {preset} | seed={cfg.SEED} | use_kd={use_kd} T={T} alpha={alpha} lr={lr} assistant_img={av if use_kd else 'n/a'}")
        print("=" * 78)
        final_results.append(run_one_kd_config(use_kd, T, alpha, lr, av, setup_id, out_dir, preset=preset))

    final_df = pd.DataFrame(final_results)[
        ["preset", "use_kd", "temperature", "alpha", "stage2_lr", "assistant_img_size", "best_epoch", "val_accuracy", "val_f1_macro", "test_accuracy", "setup_id"]
    ].reset_index(drop=True)
    final_root.mkdir(parents=True, exist_ok=True)
    summary_path = final_root / f"r8_k5_final_summary_seed_{cfg.SEED}.csv"
    final_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 78)
    print(f"R8-K5 FINAL seed {cfg.SEED} complete ({len(cfg.FINAL_SWEEP)} configs):")
    display(final_df)
    print(f"Saved summary: {summary_path}")
    print("Repeat for seeds 42, 123, 777, 2026, 3407; then aggregate TA-KD vs CE-control on test (paired t-test / McNemar).")

## Output Artifacts

Pilot: tiap config menulis checkpoint + history + metrics + prediction + confusion matrix + training curves + config JSON + artifact manifest ke `pilots/R5/<setup_id>/`, plus satu `r8_k5_pilot_sweep_summary.csv` ringkasan val. Final: artefak lengkap per seed di `final/R5/seed_<seed>/`, dengan independent test set dievaluasi setelah winner dibekukan.
